In [ ]:
from collections import deque
import heapq
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from typing import List, Tuple, Set, Dict, Optional

In [ ]:
class Graph:
    def __init__(self):
        self.graph = {}
    def add_edge(self, u, v, weight=1):
        if u not in self.graph:
            self.graph[u] = []
        self.graph[u].append((v, weight))
    def get_neighbors(self, node):
        return self.graph.get(node, [])
    def visualize(self):
        G = nx.DiGraph()
        for node, neighbors in self.graph.items():
            for neighbor, weight in neighbors:
                G.add_edge(node, neighbor, weight=weight)
        pos = nx.spring_layout(G)
        plt.figure(figsize=(12, 8))
        nx.draw(G, pos, with_labels=True, node_color='lightblue', 
                node_size=1500, font_size=16, font_weight='bold',
                arrows=True, arrowsize=20)
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(G, pos, edge_labels)
        plt.title('State Space Graph')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
g = Graph()
g.add_edge('A', 'B', 4)
g.add_edge('A', 'C', 2)
g.add_edge('B', 'D', 5)
g.add_edge('B', 'E', 10)
g.add_edge('C', 'D', 3)
g.add_edge('D', 'E', 4)
g.add_edge('E', 'F', 1)
g.add_edge('D', 'F', 7)
print("Sample State Space Graph created!")
g.visualize()

In [ ]:
def dfs(graph: Graph, start: str, goal: str) -> Tuple[List[str], int]:
    stack = [(start, [start])]
    visited = set()
    nodes_explored = 0
    while stack:
        node, path = stack.pop()
        nodes_explored += 1
        if node == goal:
            return path, nodes_explored
        if node not in visited:
            visited.add(node)
            neighbors = graph.get_neighbors(node)
            for neighbor, _ in reversed(neighbors):
                if neighbor not in visited:
                    stack.append((neighbor, path + [neighbor]))
    return None, nodes_explored
path, nodes = dfs(g, 'A', 'F')
print("\nDFS Results:")
print(f"Path found: {' -> '.join(path) if path else 'No path'}")
print(f"Nodes explored: {nodes}")
print(f"Path length: {len(path) - 1 if path else 0}")

In [ ]:
def bfs(graph: Graph, start: str, goal: str) -> Tuple[List[str], int]:
    queue = deque([(start, [start])])
    visited = {start}
    nodes_explored = 0
    while queue:
        node, path = queue.popleft()
        nodes_explored += 1
        if node == goal:
            return path, nodes_explored
        for neighbor, _ in graph.get_neighbors(node):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None, nodes_explored
path, nodes = bfs(g, 'A', 'F')
print("\nBFS Results:")
print(f"Path found: {' -> '.join(path) if path else 'No path'}")
print(f"Nodes explored: {nodes}")
print(f"Path length: {len(path) - 1 if path else 0}")

In [ ]:
test_cases = [
    ('A', 'F'),
    ('A', 'E'),
    ('B', 'F'),
    ('C', 'E')
]
results = []
for start, goal in test_cases:
    dfs_path, dfs_nodes = dfs(g, start, goal)
    bfs_path, bfs_nodes = bfs(g, start, goal)
    results.append({
        'Start-Goal': f'{start}->{goal}',
        'DFS Path Length': len(dfs_path) - 1 if dfs_path else 'N/A',
        'DFS Nodes': dfs_nodes,
        'BFS Path Length': len(bfs_path) - 1 if bfs_path else 'N/A',
        'BFS Nodes': bfs_nodes
    })
import pandas as pd
results_df = pd.DataFrame(results)
print("\nComparison of DFS and BFS:")
print(results_df.to_string(index=False))

In [ ]:
def a_star(graph: Graph, start: str, goal: str, heuristic: Dict[str, float]) -> Tuple[List[str], float, int]:
    open_set = [(heuristic.get(start, 0), 0, start, [start])]
    closed_set = set()
    nodes_explored = 0
    while open_set:
        f_score, g_score, node, path = heapq.heappop(open_set)
        nodes_explored += 1
        if node == goal:
            return path, g_score, nodes_explored
        if node in closed_set:
            continue
        closed_set.add(node)
        for neighbor, cost in graph.get_neighbors(node):
            if neighbor not in closed_set:
                new_g_score = g_score + cost
                h_score = heuristic.get(neighbor, 0)
                new_f_score = new_g_score + h_score
                heapq.heappush(open_set, (new_f_score, new_g_score, neighbor, path + [neighbor]))
    return None, float('inf'), nodes_explored
heuristic = {
    'A': 10,
    'B': 8,
    'C': 7,
    'D': 5,
    'E': 2,
    'F': 0
}
path, cost, nodes = a_star(g, 'A', 'F', heuristic)
print("\nA* Search Results:")
print(f"Path found: {' -> '.join(path) if path else 'No path'}")
print(f"Total cost: {cost}")
print(f"Nodes explored: {nodes}")
print(f"Path length: {len(path) - 1 if path else 0}")

In [ ]:
def greedy_best_first(graph: Graph, start: str, goal: str, heuristic: Dict[str, float]) -> Tuple[List[str], int]:
    open_set = [(heuristic.get(start, 0), start, [start])]
    visited = set()
    nodes_explored = 0
    while open_set:
        h_score, node, path = heapq.heappop(open_set)
        nodes_explored += 1
        if node == goal:
            return path, nodes_explored
        if node in visited:
            continue
        visited.add(node)
        for neighbor, _ in graph.get_neighbors(node):
            if neighbor not in visited:
                h_score = heuristic.get(neighbor, 0)
                heapq.heappush(open_set, (h_score, neighbor, path + [neighbor]))
    return None, nodes_explored
path, nodes = greedy_best_first(g, 'A', 'F', heuristic)
print("\nGreedy Best-First Search Results:")
print(f"Path found: {' -> '.join(path) if path else 'No path'}")
print(f"Nodes explored: {nodes}")
print(f"Path length: {len(path) - 1 if path else 0}")

In [ ]:
start_node = 'A'
goal_node = 'F'
print("\n" + "="*70)
print(f"COMPREHENSIVE ALGORITHM COMPARISON: {start_node} to {goal_node}")
print("="*70)
dfs_path, dfs_nodes = dfs(g, start_node, goal_node)
print("\n1. Depth-First Search (DFS):")
print(f"   Path: {' -> '.join(dfs_path) if dfs_path else 'No path'}")
print(f"   Nodes explored: {dfs_nodes}")
bfs_path, bfs_nodes = bfs(g, start_node, goal_node)
print("\n2. Breadth-First Search (BFS):")
print(f"   Path: {' -> '.join(bfs_path) if bfs_path else 'No path'}")
print(f"   Nodes explored: {bfs_nodes}")
greedy_path, greedy_nodes = greedy_best_first(g, start_node, goal_node, heuristic)
print("\n3. Greedy Best-First Search:")
print(f"   Path: {' -> '.join(greedy_path) if greedy_path else 'No path'}")
print(f"   Nodes explored: {greedy_nodes}")
astar_path, astar_cost, astar_nodes = a_star(g, start_node, goal_node, heuristic)
print("\n4. A* Search:")
print(f"   Path: {' -> '.join(astar_path) if astar_path else 'No path'}")
print(f"   Total cost: {astar_cost}")
print(f"   Nodes explored: {astar_nodes}")
print("\n" + "="*70)

In [ ]:
algorithms = ['DFS', 'BFS', 'Greedy', 'A*']
nodes_explored = [dfs_nodes, bfs_nodes, greedy_nodes, astar_nodes]
path_lengths = [
    len(dfs_path) - 1 if dfs_path else 0,
    len(bfs_path) - 1 if bfs_path else 0,
    len(greedy_path) - 1 if greedy_path else 0,
    len(astar_path) - 1 if astar_path else 0
]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(algorithms, nodes_explored, color=['red', 'blue', 'green', 'orange'])
ax1.set_ylabel('Nodes Explored')
ax1.set_title('Efficiency Comparison: Nodes Explored')
ax1.grid(axis='y', alpha=0.3)
ax2.bar(algorithms, path_lengths, color=['red', 'blue', 'green', 'orange'])
ax2.set_ylabel('Path Length (edges)')
ax2.set_title('Solution Quality: Path Length')
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, move=None, depth=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.depth = depth
        self.blank_pos = self.find_blank()
    def find_blank(self):
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == 0:
                    return (i, j)
        return None
    def get_neighbors(self):
        neighbors = []
        row, col = self.blank_pos
        moves = [('UP', -1, 0), ('DOWN', 1, 0), ('LEFT', 0, -1), ('RIGHT', 0, 1)]
        for move_name, dr, dc in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_board = [row[:] for row in self.board]
                new_board[row][col], new_board[new_row][new_col] = \
                    new_board[new_row][new_col], new_board[row][col]
                neighbors.append(PuzzleState(new_board, self, move_name, self.depth + 1))
        return neighbors
    def manhattan_distance(self, goal):
        distance = 0
        for i in range(3):
            for j in range(3):
                if self.board[i][j] != 0:
                    value = self.board[i][j]
                    goal_pos = None
                    for gi in range(3):
                        for gj in range(3):
                            if goal.board[gi][gj] == value:
                                goal_pos = (gi, gj)
                                break
                        if goal_pos:
                            break
                    if goal_pos:
                        distance += abs(i - goal_pos[0]) + abs(j - goal_pos[1])
        return distance
    def __eq__(self, other):
        return self.board == other.board
    def __hash__(self):
        return hash(str(self.board))
    def __lt__(self, other):
        return self.depth < other.depth
    def display(self):
        for row in self.board:
            print(' '.join(str(x) if x != 0 else '_' for x in row))
        print()
initial = PuzzleState([
    [1, 2, 3],
    [4, 0, 5],
    [7, 8, 6]
])
goal = PuzzleState([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
])
print("Initial State:")
initial.display()
print("Goal State:")
goal.display()

In [ ]:
def solve_puzzle_astar(initial, goal):
    open_set = []
    heapq.heappush(open_set, (initial.manhattan_distance(goal), initial))
    closed_set = set()
    nodes_explored = 0
    while open_set:
        _, current = heapq.heappop(open_set)
        nodes_explored += 1
        if current == goal:
            path = []
            while current:
                path.append(current)
                current = current.parent
            return list(reversed(path)), nodes_explored
        closed_set.add(current)
        for neighbor in current.get_neighbors():
            if neighbor not in closed_set:
                f_score = neighbor.depth + neighbor.manhattan_distance(goal)
                heapq.heappush(open_set, (f_score, neighbor))
    return None, nodes_explored
print("Solving 8-Puzzle with A* Search...\n")
solution, nodes = solve_puzzle_astar(initial, goal)
if solution:
    print(f"Solution found in {len(solution) - 1} moves!")
    print(f"Nodes explored: {nodes}\n")
    print("Solution steps:")
    for i, state in enumerate(solution):
        print(f"Step {i}: {state.move if state.move else 'Initial'}")
        state.display()
else:
    print("No solution found!")